<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_00_raw_data_ingestion/stage_00_raw_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_00_raw_data_preparation**

## Resumen

Esta notebook realiza la **preparación inicial del dataset intradía del MNQ (Micro E-mini Nasdaq 100)**.  
El objetivo es construir un dataset limpio, consistente y estructurado que servirá como base para la ingeniería de factores y el entrenamiento de modelos.

0. **Configuración del entorno**
   - Clonado del repositorio y montaje de Google Drive.
   - Instalación e importación de librerías necesarias.

1. **Fuente de datos**
   - Datos históricos intradía del MNQ (OHLCV, frecuencia de 1 minuto) exportados desde NinjaTrader.
   - Archivos originales en formato `.txt`, en zona horaria UTC.

2. **Generación del dataset**
   - Unificación de todos los archivos `.txt` en un único DataFrame.
   - Asignación de nombres de columnas: `open`, `high`, `low`, `close`, `volume`.
   - Conversión de la columna `datetime` a índice temporal.

3. **Filtrado**
   - Conserva solo **días hábiles bursátiles** (se eliminan fines de semana y feriados de mercado de EE.UU.).
   - Conversión de marcas de tiempo de **UTC → US/Eastern**.
   - Filtrado de **horario de mercado** (09:30–16:00) más pre-market (desde 08:30).

4. **Validación de registros diarios**
   - Verificación de que cada día contenga la cantidad esperada de registros minuto a minuto.
   - Detección y eliminación de días incompletos o con irregularidades.

5. **Chequeo de continuidad temporal**
   - Confirmación de que los datos intradía estén en intervalos consecutivos de 1 minuto, sin gaps.

6. **Dataset final**
   - Guardado del dataset limpio en formato `.parquet` dentro de Google Drive.

---

**Resultado:** Un dataset intradía del MNQ completamente limpio y estandarizado, listo para la ingeniería de factores y el modelado.

## 0. Configuración del Entorno

### 0.1. Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías

In [2]:
import sys
!{sys.executable} -m pip install -q pandas_market_calendars
print("✅ Librería instalada: pandas_market_calendars")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.3/213.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.
✅ Librería instalada: pandas_market_calendars


### 0.3. Importación de librerías

In [3]:
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
import pandas as pd
from tabulate import tabulate

# Calendario de mercados
import pandas_market_calendars as mcal
import pandas as pd
import requests
from io import StringIO


# **1. Contexto y fuente de datos**

Los datos corresponden al contrato MNQ (Micro E-mini Nasdaq 100) descargados desde NinjaTrader con frecuencia de un minuto (formato OHLCV).

- Open: precio de apertura
- High: precio máximo
- Low: precio mínimo
- Close: precio de cierre
- Volume: volumen negociado

Los datos están en la zona horaria UTC.


In [12]:
import glob
import pandas as pd

def analizar_rangos_txt(drive_path):

    ruta_historicos = f'{drive_path}/data/00_source/*.txt'
    archivos = glob.glob(ruta_historicos)

    if not archivos:
        raise FileNotFoundError("No se encontraron archivos históricos.")

    resumen = []

    for archivo in sorted(archivos):

        df = pd.read_csv(
            archivo,
            sep=';',
            header=None,
            usecols=[0],  # SOLO datetime (más eficiente)
            names=['datetime']
        )

        df['datetime'] = pd.to_datetime(df['datetime'], format='%Y%m%d %H%M%S')

        fecha_inicio = df['datetime'].min()
        fecha_fin = df['datetime'].max()
        n_rows = len(df)

        resumen.append({
            "archivo": archivo.split("/")[-1],
            "fecha_inicio": fecha_inicio,
            "fecha_fin": fecha_fin,
            "n_rows": n_rows
        })

    df_resumen = pd.DataFrame(resumen).sort_values("fecha_inicio")

    return df_resumen

In [9]:
df_rangos = analizar_rangos_txt(drive_path)
df_rangos

FileNotFoundError: No se encontraron archivos históricos.

In [10]:
import glob
import os
import pandas as pd

def analizar_superposicion_txt(drive_path, filtro_especial="06_25"):

    ruta_historicos = f"{drive_path}/data/00_source/*.txt"
    archivos = sorted(glob.glob(ruta_historicos))

    if not archivos:
        raise FileNotFoundError("No se encontraron archivos históricos.")

    resumen = []

    # =========================
    # 1. Leer rango de cada archivo
    # =========================
    for archivo in archivos:
        df = pd.read_csv(
            archivo,
            sep=";",
            header=None,
            usecols=[0],
            names=["datetime"]
        )

        df["datetime"] = pd.to_datetime(df["datetime"], format="%Y%m%d %H%M%S")

        resumen.append({
            "archivo": os.path.basename(archivo),
            "ruta": archivo,
            "fecha_inicio": df["datetime"].min(),
            "fecha_fin": df["datetime"].max(),
            "n_rows": len(df),
            "archivo_especial": filtro_especial in os.path.basename(archivo)
        })

    df_rangos = pd.DataFrame(resumen).sort_values(["fecha_inicio", "fecha_fin"]).reset_index(drop=True)

    # =========================
    # 2. Buscar superposición entre archivos
    # =========================
    superposiciones = []

    for i in range(len(df_rangos)):
        archivo_i = df_rangos.loc[i]

        for j in range(i + 1, len(df_rangos)):
            archivo_j = df_rangos.loc[j]

            # Si el siguiente empieza después de que termina el actual,
            # ya no puede superponerse con este ni con los siguientes
            if archivo_j["fecha_inicio"] > archivo_i["fecha_fin"]:
                break

            # Verificar intersección
            inicio_solape = max(archivo_i["fecha_inicio"], archivo_j["fecha_inicio"])
            fin_solape = min(archivo_i["fecha_fin"], archivo_j["fecha_fin"])

            if inicio_solape <= fin_solape:
                superposiciones.append({
                    "archivo_1": archivo_i["archivo"],
                    "archivo_2": archivo_j["archivo"],
                    "inicio_solape": inicio_solape,
                    "fin_solape": fin_solape,
                    "duracion_solape": fin_solape - inicio_solape,
                    "archivo_1_especial": archivo_i["archivo_especial"],
                    "archivo_2_especial": archivo_j["archivo_especial"]
                })

    df_super = pd.DataFrame(superposiciones)

    # =========================
    # 3. Filtrar superposiciones donde aparezca 06_25
    # =========================
    if not df_super.empty:
        df_super_0625 = df_super[
            (df_super["archivo_1_especial"]) | (df_super["archivo_2_especial"])
        ].copy()
    else:
        df_super_0625 = pd.DataFrame()

    return df_rangos, df_super, df_super_0625

In [11]:
df_rangos, df_super, df_super_0625 = analizar_superposicion_txt(drive_path, filtro_especial="06_25")

FileNotFoundError: No se encontraron archivos históricos.

In [ ]:
df_rangos

,archivo,ruta,fecha_inicio,fecha_fin,n_rows,archivo_especial
0,00_mnq_03_20.Last.txt,/content/drive/MyDrive/neural_profit/data/00_s...,2019-12-23 03:01:00,2020-03-20 13:30:00,81660,False
1,01_mnq_06_20.Last.txt,/content/drive/MyDrive/neural_profit/data/00_s...,2020-03-23 03:01:00,2020-06-19 13:30:00,86069,False
2,02_mnq_09_20.Last.txt,/content/drive/MyDrive/neural_profit/data/00_s...,2020-06-22 03:01:00,2020-09-18 13:30:00,87482,False
3,03_mnq_12_20.Last.txt,/content/drive/MyDrive/neural_profit/data/00_s...,2020-09-21 03:01:00,2020-12-18 03:00:00,86824,False
4,04_mnq_03_21.Last.txt,/content/drive/MyDrive/neural_profit/data/00_s...,2020-12-21 03:01:00,2021-03-19 13:30:00,84599,False
5,05_mnq_06_21.Last.txt,/content/drive/MyDrive/neural_profit/data/00_s...,2021-03-22 03:01:00,2021-06-18 13:30:00,87090,False
6,06_mnq_09_21.Last.txt,/content/drive/MyDrive/neural_profit/data/00_s...,2021-06-21 03:01:00,2021-09-17 13:30:00,88205,False
7,07_mnq_12_21.Last.txt,/content/drive/MyDrive/neural_profit/data/00_s...,2021-09-20 03:01:00,2021-12-17 14:30:00,88318,False
8,08_mnq_03_22.Last.txt,/content/drive/MyDrive/neural_profit/data/00_s...,2021-12-20 03:01:00,2022-03-18 13:30:00,87112,False
9,09_mnq_06_22.Last.txt,/content/drive/MyDrive/neural_profit/data/00_s...,2022-03-21 03:01:00,2022-06-17 13:30:00,87336,False


In [ ]:
df_super

""


# **2. Generación de dataset desde archivos históricos**

Dado que los contratos se encuentran almacenados en archivos .txt dentro de la carpeta historicos_mnq, es necesario unificarlos en un único dataset consolidado.

La siguiente función se encarga de leer los archivos .txt, asignar nombres a las columnas correspondientes y establecer la columna datetime como índice temporal del dataframe.

In [6]:
def generar_df ():

    # Ruta a los archivos .txt
    ruta_historicos_drive = f'{drive_path}/data/00_source/*.txt'

    # Determinar qué ruta usar
    if glob.glob(ruta_historicos_drive):
        print("Usando históricos desde Google Drive")
        ruta_historicos = ruta_historicos_drive
    else:
        raise FileNotFoundError("No se encontraron archivos históricos en el Drive.")

    # Lista para almacenar DataFrames individuales
    df_mnq = []

    # Leer todos los archivos .txt
    for archivo in glob.glob(ruta_historicos):
        df = pd.read_csv(
            archivo,
            sep=';',
            header=None,
            names=['datetime', 'open', 'high', 'low', 'close', 'volume'],
            dtype={'open': float, 'high': float, 'low': float, 'close': float, 'volume': int}
        )

        # Convertir columna 'datetime' al formato datetime real
        df['datetime'] = pd.to_datetime(df['datetime'], format='%Y%m%d %H%M%S')

        # Establecer como índice
        df.set_index('datetime', inplace=True)

        df_mnq.append(df)

    # Unir todos los DataFrames
    df_mnq_raw = pd.concat(df_mnq)
    # Ordenar por fecha si es necesario
    df_mnq_raw.sort_index(inplace=True)

    return df_mnq_raw

El siguiente bloque de código verifica si el dataset consolidado ya ha sido generado previamente.

En particular, comprueba la existencia del archivo mnq_raw.parquet.

- Si el archivo está presente, se carga directamente en la variable df_mnq.

- En caso contrario, se invoca la función generate_dataset() para generar el dataset a partir de los archivos originales.

In [7]:
import os
import pandas as pd

def load_or_build_raw_dataset():

    raw_dir = f"{drive_path}/data/01_raw"
    mnq_raw_data_file = f"{raw_dir}/mnq_raw.parquet"

    # Crear carpeta si no existe
    os.makedirs(raw_dir, exist_ok=True)

    if os.path.exists(mnq_raw_data_file):
        print("Archivo encontrado en disco. Cargando dataset local...")
        df_mnq_raw = pd.read_parquet(mnq_raw_data_file)

    else:
        print("Archivo no encontrado. Generando dataset desde archivos históricos...")
        df_mnq_raw = generar_df()
        df_mnq_raw.to_parquet(mnq_raw_data_file, index=True)
        print("Dataset generado y guardado localmente.")

    return df_mnq_raw

In [8]:
df_mnq_raw = load_or_build_raw_dataset()

Archivo no encontrado. Generando dataset desde archivos históricos...


FileNotFoundError: No se encontraron archivos históricos en el Drive.

In [ ]:
df_mnq_raw

,open,high,low,close,volume
datetime,,,,,
2019-12-23 03:01:00,8718.50,8718.75,8718.50,8718.50,9
2019-12-23 03:02:00,8718.25,8718.25,8718.00,8718.25,14
2019-12-23 03:03:00,8718.25,8718.50,8718.00,8718.25,74
2019-12-23 03:04:00,8718.25,8719.00,8718.25,8718.50,10
2019-12-23 03:05:00,8718.50,8719.00,8718.50,8719.00,6
...,...,...,...,...,...
2026-04-17 20:14:00,26835.50,26844.50,26835.50,26839.25,1120
2026-04-17 20:15:00,26839.50,26842.25,26837.75,26840.25,605
2026-04-17 20:16:00,26840.75,26841.75,26835.75,26841.25,615


In [ ]:
# Duplicados exactos de índice (timestamp)
duplicados = df_mnq_raw.index.duplicated(keep=False)

df_dup = df_mnq_raw[duplicados]

print(f"Total registros duplicados: {duplicados.sum()}")

df_dup.head()

Total registros duplicados: 0


,open,high,low,close,volume
datetime,,,,,


In [ ]:
# Crear columna fecha
df_tmp = df_mnq_raw.copy()
df_tmp["date"] = df_tmp.index.date

# Contar registros por día
conteo = df_tmp.groupby("date").size()

# Días sospechosos (más de lo esperado)
# En MNQ intradía suelen ser ~390–450 registros por día
dias_sospechosos = conteo[conteo > 500]

dias_sospechosos.sort_values(ascending=False)

,0
date,
2025-01-30,1382
2025-11-19,1382
2026-01-12,1381
2025-02-06,1381
2024-07-17,1381
...,...
2025-07-15,668
2021-06-18,653
2020-03-16,642


In [ ]:
df_tmp = df_mnq_raw.copy()
df_tmp["date"] = df_tmp.index.date

duplicados_por_dia = (
    df_tmp
    .reset_index()
    .groupby(["date", "datetime"])
    .size()
    .reset_index(name="count")
)

# Filtrar donde hay más de 1 ocurrencia del mismo timestamp
duplicados_reales = duplicados_por_dia[duplicados_por_dia["count"] > 1]

duplicados_reales.head()

,date,datetime,count


In [ ]:
n_timestamps_duplicados = duplicados_reales.shape[0]
n_dias_afectados = duplicados_reales["date"].nunique()

print(f"Timestamps duplicados: {n_timestamps_duplicados}")
print(f"Días con duplicados: {n_dias_afectados}")

Timestamps duplicados: 0
Días con duplicados: 0
